# CatAPI RAG Inspection

Учебный notebook для инспекции CatAPI RAG в проекте `cat-breed-assistant`.

Данные уже подготовлены заранее и лежат в репозитории, поэтому notebook **не скачивает CatAPI**, **не использует внешние датасеты** и **не чинит Kaggle-зависимости**.

Пайплайн:

```text
catapi_breed_documents.jsonl → catapi_chunks.jsonl → embeddings → ChromaDB → retrieval → RAG prompt → answer
```

## 1. Цель notebook

CatAPI RAG Inspection Notebook

Цель ноутбука — пошагово проверить RAG-пайплайн для Cat Breed Assistant.

В этой версии мы не скачиваем данные из CatAPI и не используем Hugging Face dataset.

У нас уже есть подготовленные файлы:

- `catapi_breed_documents.jsonl`
- `catapi_chunks.jsonl`

Проверяем путь:

`processed documents → chunks → retrieval → RAG prompt → answer`

Сначала делаем максимально устойчивую версию без pandas, ChromaDB и sentence-transformers, чтобы не ловить конфликты зависимостей в Kaggle.

In [1]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/MataNerdy/cat-breed-assistant.git"
KAGGLE_WORKING = Path("/kaggle/working")
KAGGLE_PROJECT_DIR = KAGGLE_WORKING / "cat-breed-assistant"

if KAGGLE_WORKING.exists():
    if KAGGLE_PROJECT_DIR.exists():
        print(f"Repository already exists: {KAGGLE_PROJECT_DIR}")
    else:
        print(f"Cloning {REPO_URL} into {KAGGLE_PROJECT_DIR}...")
        subprocess.run(
            ["git", "clone", REPO_URL, str(KAGGLE_PROJECT_DIR)],
            check=True,
        )
        print(f"Cloned repository to {KAGGLE_PROJECT_DIR}")
else:
    print("Not running in Kaggle /kaggle/working environment. Skipping clone cell.")

Cloning https://github.com/MataNerdy/cat-breed-assistant.git into /kaggle/working/cat-breed-assistant...


Cloning into '/kaggle/working/cat-breed-assistant'...


Cloned repository to /kaggle/working/cat-breed-assistant


In [6]:
from pathlib import Path
import json
import os
import re
import textwrap
from collections import Counter
from typing import Any, Dict, List, Optional

In [7]:
def find_file(filename: str, search_roots=None) -> Optional[Path]:
    """
    Ищет файл в типичных местах:
    - текущая папка
    - data/processed/
    - /kaggle/input/
    - /mnt/data/
    """
    if search_roots is None:
        search_roots = [
            Path.cwd(),
            Path.cwd() / "data" / "processed",
            Path("/kaggle/input"),
            Path("/mnt/data"),
        ]

    for root in search_roots:
        if not root.exists():
            continue

        direct = root / filename
        if direct.exists():
            return direct

        # recursive search, useful for Kaggle datasets
        try:
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]
        except Exception:
            pass

    return None


DOCS_PATH = find_file("catapi_breed_documents.jsonl")
CHUNKS_PATH = find_file("catapi_chunks.jsonl")

print("Current working directory:", Path.cwd())
print("Documents path:", DOCS_PATH)
print("Chunks path:", CHUNKS_PATH)

if DOCS_PATH is None:
    print("Не найден catapi_breed_documents.jsonl")

if CHUNKS_PATH is None:
    print("Не найден catapi_chunks.jsonl")

Current working directory: /kaggle/working/cat-breed-assistant
Documents path: /kaggle/working/cat-breed-assistant/data/processed/catapi_breed_documents.jsonl
Chunks path: /kaggle/working/cat-breed-assistant/data/processed/catapi_chunks.jsonl


In [8]:
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def preview_text(text: str, n: int = 300) -> str:
    if text is None:
        return ""
    text = str(text).replace("\n", " ").strip()
    return text[:n] + ("..." if len(text) > n else "")


def show_json(obj: Any, max_chars: int = 4000):
    text = json.dumps(obj, ensure_ascii=False, indent=2)
    if len(text) > max_chars:
        text = text[:max_chars] + "\n..."
    print(text)


def show_rows(rows: List[Dict[str, Any]], limit: int = 10):
    """
    Простой вывод таблицы без pandas.
    """
    for i, row in enumerate(rows[:limit], start=1):
        print("=" * 80)
        print(f"ROW {i}")
        show_json(row, max_chars=2500)

In [9]:
if DOCS_PATH is None or CHUNKS_PATH is None:
    raise FileNotFoundError("Нужны оба файла: catapi_breed_documents.jsonl и catapi_chunks.jsonl")

documents = load_jsonl(DOCS_PATH)
chunks = load_jsonl(CHUNKS_PATH)

print("Documents:", len(documents))
print("Chunks:", len(chunks))

doc_breeds = sorted({d.get("breed_name") for d in documents})
chunk_breeds = sorted({c.get("breed_name") for c in chunks})

print("Unique breeds in documents:", len(doc_breeds))
print("Unique breeds in chunks:", len(chunk_breeds))

print("\nFirst 10 breeds:")
for name in chunk_breeds[:10]:
    print("-", name)

Documents: 67
Chunks: 67
Unique breeds in documents: 67
Unique breeds in chunks: 67

First 10 breeds:
- Abyssinian
- Aegean
- American Bobtail
- American Curl
- American Shorthair
- American Wirehair
- Arabian Mau
- Australian Mist
- Balinese
- Bambino


In [10]:
print("Document keys:")
print(documents[0].keys())

print("\nDocument example:")
show_json(documents[0], max_chars=3000)

Document keys:
dict_keys(['doc_id', 'breed_id', 'breed_name', 'source', 'text', 'metadata'])

Document example:
{
  "doc_id": "catapi_abys",
  "breed_id": "abys",
  "breed_name": "Abyssinian",
  "source": "thecatapi",
  "text": "Breed name: Abyssinian\nDescription: The Abyssinian is easy to care for, and a joy to have in your home. They’re affectionate cats and love both people and other animals.\nTemperament: Active, Energetic, Independent, Intelligent, Gentle\nOrigin: Egypt\nLife span: 14 - 15 years\nWeight: 3 - 5 kg (7  -  10 lb)\nGrooming level: 1\nEnergy level: 5\nHealth issues score: 2\nChild friendly score: 3\nDog friendly score: 4\nStranger friendly score: 5\nIntelligence score: 5\nHairless: no\nShedding level: 2\nSocial needs score: 5\nVocalisation score: 1\nHypoallergenic: 0\nWikipedia URL: https://en.wikipedia.org/wiki/Abyssinian_(cat)",
  "metadata": {
    "origin": "Egypt",
    "temperament": "Active, Energetic, Independent, Intelligent, Gentle",
    "life_span": "14 - 15"

In [11]:
print("Chunk keys:")
print(chunks[0].keys())

print("\nChunk example:")
show_json(chunks[0], max_chars=3000)

Chunk keys:
dict_keys(['chunk_id', 'doc_id', 'breed_id', 'breed_name', 'source', 'text', 'metadata'])

Chunk example:
{
  "chunk_id": "catapi_abys_profile",
  "doc_id": "catapi_abys",
  "breed_id": "abys",
  "breed_name": "Abyssinian",
  "source": "thecatapi",
  "text": "Breed name: Abyssinian\nDescription: The Abyssinian is easy to care for, and a joy to have in your home. They’re affectionate cats and love both people and other animals.\nTemperament: Active, Energetic, Independent, Intelligent, Gentle\nOrigin: Egypt\nLife span: 14 - 15 years\nWeight: 3 - 5 kg (7  -  10 lb)\nGrooming level: 1\nEnergy level: 5\nHealth issues score: 2\nChild friendly score: 3\nDog friendly score: 4\nStranger friendly score: 5\nIntelligence score: 5\nHairless: no\nShedding level: 2\nSocial needs score: 5\nVocalisation score: 1\nHypoallergenic: 0\nWikipedia URL: https://en.wikipedia.org/wiki/Abyssinian_(cat)",
  "metadata": {
    "breed_id": "abys",
    "breed_name": "Abyssinian",
    "source": "thecatapi

In [12]:
overview_rows = []

for c in chunks:
    metadata = c.get("metadata", {}) or {}
    overview_rows.append({
        "breed_id": c.get("breed_id"),
        "breed_name": c.get("breed_name"),
        "origin": metadata.get("origin"),
        "wikipedia_url": metadata.get("wikipedia_url"),
        "reference_image_id": metadata.get("reference_image_id"),
        "text_preview": preview_text(c.get("text", ""), 180),
    })

show_rows(overview_rows, limit=5)

ROW 1
{
  "breed_id": "abys",
  "breed_name": "Abyssinian",
  "origin": "Egypt",
  "wikipedia_url": "https://en.wikipedia.org/wiki/Abyssinian_(cat)",
  "reference_image_id": "0XYvRd7oD",
  "text_preview": "Breed name: Abyssinian Description: The Abyssinian is easy to care for, and a joy to have in your home. They’re affectionate cats and love both people and other animals. Temperamen..."
}
ROW 2
{
  "breed_id": "aege",
  "breed_name": "Aegean",
  "origin": "Greece",
  "wikipedia_url": "https://en.wikipedia.org/wiki/Aegean_cat",
  "reference_image_id": "ozEvzdVM-",
  "text_preview": "Breed name: Aegean Description: Native to the Greek islands known as the Cyclades in the Aegean Sea, these are natural cats, meaning they developed without humans getting involved ..."
}
ROW 3
{
  "breed_id": "abob",
  "breed_name": "American Bobtail",
  "origin": "United States",
  "wikipedia_url": "https://en.wikipedia.org/wiki/American_Bobtail",
  "reference_image_id": "hBXicehMA",
  "text_preview": "Bre

In [13]:
def find_breed_chunk(breed_query: str) -> Optional[Dict[str, Any]]:
    q = breed_query.lower().strip()
    for chunk in chunks:
        name = str(chunk.get("breed_name", "")).lower()
        breed_id = str(chunk.get("breed_id", "")).lower()
        if q == name or q == breed_id or q in name:
            return chunk
    return None


for breed in ["British Shorthair", "Maine Coon", "Siamese", "Persian", "Sphynx"]:
    print("=" * 100)
    print(breed)
    chunk = find_breed_chunk(breed)

    if chunk is None:
        print("Не найдено")
        continue

    metadata = chunk.get("metadata", {}) or {}

    print("breed_id:", chunk.get("breed_id"))
    print("origin:", metadata.get("origin"))
    print("wikipedia_url:", metadata.get("wikipedia_url"))
    print("reference_image_id:", metadata.get("reference_image_id"))
    print()
    print(preview_text(chunk.get("text", ""), 1200))

British Shorthair
breed_id: bsho
origin: United Kingdom
wikipedia_url: https://en.wikipedia.org/wiki/British_Shorthair
reference_image_id: s4wQfYoEk

Breed name: British Shorthair Description: The British Shorthair is a very pleasant cat to have as a companion, ans is easy going and placid. The British is a fiercely loyal, loving cat and will attach herself to every one of her family members. While loving to play, she doesn't need hourly attention. If she is in the mood to play, she will find someone and bring a toy to that person. The British also plays well by herself, and thus is a good companion for single people. Temperament: Affectionate, Easy Going, Gentle, Loyal, Patient, calm Origin: United Kingdom Life span: 12 - 17 years Weight: 5 - 9 kg (12 - 20 lb) Grooming level: 2 Energy level: 2 Health issues score: 2 Child friendly score: 4 Dog friendly score: 5 Stranger friendly score: 2 Intelligence score: 3 Hairless: no Shedding level: 4 Social needs score: 3 Vocalisation score: 1 H

## Первый retrieval без embeddings

Пока не используем ChromaDB и sentence-transformers.

Сначала сделаем простой retrieval:

1. определяем породу по русским и английским alias;
2. если порода найдена — возвращаем chunk этой породы;
3. если порода не найдена — используем простой keyword scoring.

Это не финальный semantic RAG, но это устойчивый baseline, который не зависит от Kaggle-окружения.

In [14]:
BREED_ALIASES = {
    "британ": "British Shorthair",
    "британск": "British Shorthair",
    "british": "British Shorthair",
    "shorthair": "British Shorthair",

    "мейн": "Maine Coon",
    "мейн-кун": "Maine Coon",
    "мейн кун": "Maine Coon",
    "maine": "Maine Coon",
    "coon": "Maine Coon",

    "сфинкс": "Sphynx",
    "sphynx": "Sphynx",
    "hairless": "Sphynx",

    "сиам": "Siamese",
    "siamese": "Siamese",

    "перс": "Persian",
    "персид": "Persian",
    "persian": "Persian",

    "бенгал": "Bengal",
    "bengal": "Bengal",

    "рэгдолл": "Ragdoll",
    "ragdoll": "Ragdoll",

    "абиссин": "Abyssinian",
    "abyssinian": "Abyssinian",
}


def detect_breed_alias(question: str) -> Optional[str]:
    q = question.lower()
    for alias, breed_name in BREED_ALIASES.items():
        if alias in q:
            return breed_name
    return None


test_questions = [
    "Чем британские котики отличаются от обычных?",
    "Расскажи про мейн куна",
    "Как ухаживать за сфинксом?",
    "Сиамская кошка разговорчивая?",
    "Persian cat grooming",
]

for q in test_questions:
    print(q, "→", detect_breed_alias(q))

Чем британские котики отличаются от обычных? → British Shorthair
Расскажи про мейн куна → Maine Coon
Как ухаживать за сфинксом? → Sphynx
Сиамская кошка разговорчивая? → Siamese
Persian cat grooming → Persian


In [15]:
STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "with", "cat", "cats",
    "кошка", "кошки", "кот", "коты", "котик", "котики", "про", "как", "чем",
    "что", "это", "у", "за", "и", "в", "на", "от", "для"
}


def tokenize(text: str) -> List[str]:
    text = text.lower()
    text = re.sub(r"[^a-zа-яё0-9\s-]", " ", text)
    tokens = [t.strip() for t in text.split()]
    return [t for t in tokens if len(t) > 2 and t not in STOPWORDS]


def keyword_score(query: str, text: str) -> int:
    q_tokens = set(tokenize(query))
    t_tokens = set(tokenize(text))
    return len(q_tokens & t_tokens)


def keyword_retrieve(question: str, top_k: int = 5) -> List[Dict[str, Any]]:
    scored = []

    for chunk in chunks:
        score = keyword_score(question, chunk.get("text", ""))
        scored.append({
            "score": score,
            "chunk": chunk,
        })

    scored = sorted(scored, key=lambda x: x["score"], reverse=True)
    return scored[:top_k]

In [16]:
def hybrid_retrieve(question: str, top_k: int = 5) -> Dict[str, Any]:
    detected_breed = detect_breed_alias(question)

    if detected_breed:
        breed_chunk = find_breed_chunk(detected_breed)
        if breed_chunk:
            return {
                "strategy": "alias_exact_breed",
                "detected_breed": detected_breed,
                "results": [
                    {
                        "rank": 1,
                        "score": None,
                        "chunk": breed_chunk,
                    }
                ],
            }

    keyword_results = keyword_retrieve(question, top_k=top_k)

    return {
        "strategy": "keyword_fallback",
        "detected_breed": detected_breed,
        "results": [
            {
                "rank": i + 1,
                "score": item["score"],
                "chunk": item["chunk"],
            }
            for i, item in enumerate(keyword_results)
        ],
    }


def print_retrieval_result(question: str, result: Dict[str, Any]):
    print("=" * 100)
    print("QUESTION:", question)
    print("STRATEGY:", result["strategy"])
    print("DETECTED BREED:", result["detected_breed"])
    print()

    for item in result["results"]:
        chunk = item["chunk"]
        metadata = chunk.get("metadata", {}) or {}

        print("-" * 80)
        print("rank:", item["rank"])
        print("score:", item["score"])
        print("breed:", chunk.get("breed_name"))
        print("breed_id:", chunk.get("breed_id"))
        print("origin:", metadata.get("origin"))
        print("wikipedia_url:", metadata.get("wikipedia_url"))
        print("reference_image_id:", metadata.get("reference_image_id"))
        print("text:", preview_text(chunk.get("text", ""), 500))

In [17]:
queries = [
    "Чем британские котики отличаются от обычных?",
    "Расскажи про мейн куна",
    "Как ухаживать за сфинксом?",
    "Сиамская кошка разговорчивая и умная?",
    "Persian cat grooming",
    "large gentle cat",
    "round face dense plush coat",
]

for q in queries:
    result = hybrid_retrieve(q, top_k=5)
    print_retrieval_result(q, result)

QUESTION: Чем британские котики отличаются от обычных?
STRATEGY: alias_exact_breed
DETECTED BREED: British Shorthair

--------------------------------------------------------------------------------
rank: 1
score: None
breed: British Shorthair
breed_id: bsho
origin: United Kingdom
wikipedia_url: https://en.wikipedia.org/wiki/British_Shorthair
reference_image_id: s4wQfYoEk
text: Breed name: British Shorthair Description: The British Shorthair is a very pleasant cat to have as a companion, ans is easy going and placid. The British is a fiercely loyal, loving cat and will attach herself to every one of her family members. While loving to play, she doesn't need hourly attention. If she is in the mood to play, she will find someone and bring a toy to that person. The British also plays well by herself, and thus is a good companion for single people. Temperament: Affectionat...
QUESTION: Расскажи про мейн куна
STRATEGY: alias_exact_breed
DETECTED BREED: Maine Coon

--------------------------

## Что мы уже проверили

На этом этапе мы ещё не использовали embeddings и ChromaDB.

Но мы уже проверили:

- данные загружаются из JSONL;
- в данных есть 67 пород;
- у каждой породы есть текстовый профиль;
- русские запросы можно связать с породой через alias detection;
- retrieved context можно явно показать перед генерацией ответа.

Это уже минимальный retrieval layer.

Следующий шаг — собрать RAG prompt.

In [18]:
SYSTEM_INSTRUCTION = """
Ты — дружелюбный ассистент-консультант по породам кошек.

Отвечай на русском языке.
Используй только факты из retrieved context.
Если информации мало, честно скажи, что данных недостаточно.
Не давай категоричных медицинских советов.
Если вопрос касается здоровья, мягко посоветуй обратиться к ветеринару.
""".strip()


def build_rag_prompt(question: str, retrieved_result: Dict[str, Any]) -> str:
    context_blocks = []

    for item in retrieved_result["results"]:
        chunk = item["chunk"]
        metadata = chunk.get("metadata", {}) or {}

        block = f"""
[Context chunk]
Breed: {chunk.get("breed_name")}
Breed ID: {chunk.get("breed_id")}
Origin: {metadata.get("origin")}
Wikipedia URL: {metadata.get("wikipedia_url")}
Reference image ID: {metadata.get("reference_image_id")}

Text:
{chunk.get("text")}
""".strip()

        context_blocks.append(block)

    context_text = "\n\n---\n\n".join(context_blocks)

    prompt = f"""
SYSTEM INSTRUCTION:
{SYSTEM_INSTRUCTION}

RETRIEVED CONTEXT:
{context_text}

USER QUESTION:
{question}

ANSWER:
""".strip()

    return prompt


question = "Чем британские котики отличаются от обычных?"
retrieved = hybrid_retrieve(question, top_k=5)
prompt = build_rag_prompt(question, retrieved)

print(prompt)

SYSTEM INSTRUCTION:
Ты — дружелюбный ассистент-консультант по породам кошек.

Отвечай на русском языке.
Используй только факты из retrieved context.
Если информации мало, честно скажи, что данных недостаточно.
Не давай категоричных медицинских советов.
Если вопрос касается здоровья, мягко посоветуй обратиться к ветеринару.

RETRIEVED CONTEXT:
[Context chunk]
Breed: British Shorthair
Breed ID: bsho
Origin: United Kingdom
Wikipedia URL: https://en.wikipedia.org/wiki/British_Shorthair
Reference image ID: s4wQfYoEk

Text:
Breed name: British Shorthair
Description: The British Shorthair is a very pleasant cat to have as a companion, ans is easy going and placid. The British is a fiercely loyal, loving cat and will attach herself to every one of her family members. While loving to play, she doesn't need hourly attention. If she is in the mood to play, she will find someone and bring a toy to that person. The British also plays well by herself, and thus is a good companion for single people.


In [19]:
def make_mock_rag_answer(question: str, retrieved_result: Dict[str, Any]) -> str:
    if not retrieved_result["results"]:
        return "Я не нашла подходящий контекст по породам кошек."

    chunk = retrieved_result["results"][0]["chunk"]
    metadata = chunk.get("metadata", {}) or {}

    breed_name = chunk.get("breed_name")
    origin = metadata.get("origin")
    wiki = metadata.get("wikipedia_url")
    ref_image = metadata.get("reference_image_id")

    text = chunk.get("text", "")

    answer = f"""
По найденному контексту речь, скорее всего, про породу **{breed_name}**.

Коротко по данным CatAPI:

{preview_text(text, 900)}

Источник указывает происхождение: **{origin}**.

Это не медицинская консультация: если вопрос касается здоровья, лучше свериться с ветеринаром.

Wikipedia URL: {wiki}
Reference image ID: {ref_image}
""".strip()

    return answer


print(make_mock_rag_answer(question, retrieved))

По найденному контексту речь, скорее всего, про породу **British Shorthair**.

Коротко по данным CatAPI:

Breed name: British Shorthair Description: The British Shorthair is a very pleasant cat to have as a companion, ans is easy going and placid. The British is a fiercely loyal, loving cat and will attach herself to every one of her family members. While loving to play, she doesn't need hourly attention. If she is in the mood to play, she will find someone and bring a toy to that person. The British also plays well by herself, and thus is a good companion for single people. Temperament: Affectionate, Easy Going, Gentle, Loyal, Patient, calm Origin: United Kingdom Life span: 12 - 17 years Weight: 5 - 9 kg (12 - 20 lb) Grooming level: 2 Energy level: 2 Health issues score: 2 Child friendly score: 4 Dog friendly score: 5 Stranger friendly score: 2 Intelligence score: 3 Hairless: no Shedding level: 4 Social needs score: 3 Vocalisation score: 1 Hypoallergenic: 0 Wikipedia URL: https://en.w

## Промежуточные выводы

Что получилось:

- мы загрузили готовые CatAPI documents/chunks;
- проверили, что данные содержат десятки пород;
- сделали простой retrieval без тяжёлых зависимостей;
- для русских запросов использовали alias detection;
- собрали RAG prompt;
- получили mock answer на основе retrieved context.

Что важно:

- это ещё не vector RAG;
- зато это уже контролируемый retrieval pipeline;
- он не зависит от нестабильного Kaggle окружения;
- теперь понятно, какие факты реально попадают в ответ.

Следующий этап:

1. добавить embeddings, если окружение позволит;
2. попробовать multilingual embedding model;
3. подключить этот CatAPI retriever к backend;
4. показать retrieved context и картинки в Streamlit.

## Диагностика baseline retrieval

Что видно по результатам:

1. Запросы с явным названием породы работают хорошо через alias detection.
2. Keyword fallback работает слабо:
   - он не понимает смысл;
   - он не умеет работать с признаками вроде "большая", "спокойная", "длинная шерсть";
   - он часто выбирает случайную породу с одним совпавшим словом.

Поэтому перед embeddings сделаем более умный baseline:

`query → alias detection → structured CatAPI fields → ranked breeds`

Будем использовать числовые и текстовые признаки из CatAPI:
- grooming level;
- energy level;
- shedding level;
- vocalisation score;
- social needs;
- hairless;
- weight;
- temperament;
- description.

In [25]:
def extract_field(text: str, field_name: str) -> Optional[str]:
    """
    Достаёт значение поля из текстового профиля вида:
    Field name: value
    """
    pattern = rf"{re.escape(field_name)}:\s*(.*?)(?=\n[A-Z][A-Za-z ]+?:|\Z)"
    match = re.search(pattern, text, flags=re.S)
    if not match:
        return None
    return match.group(1).strip()


def extract_int_field(text: str, field_name: str) -> Optional[int]:
    value = extract_field(text, field_name)
    if value is None:
        return None
    match = re.search(r"\d+", value)
    return int(match.group(0)) if match else None


def parse_weight_kg(text: str) -> Optional[tuple]:
    """
    Достаёт вес в кг из строки вроде:
    Weight: 5 - 9 kg (12 - 20 lb)
    """
    value = extract_field(text, "Weight")
    if not value:
        return None

    match = re.search(r"(\d+)\s*-\s*(\d+)\s*kg", value)
    if not match:
        return None

    return int(match.group(1)), int(match.group(2))


def build_structured_profile(chunk: Dict[str, Any]) -> Dict[str, Any]:
    text = chunk.get("text", "")
    metadata = chunk.get("metadata", {}) or {}

    weight_kg = parse_weight_kg(text)

    return {
        "breed_id": chunk.get("breed_id"),
        "breed_name": chunk.get("breed_name"),
        "origin": metadata.get("origin"),
        "description": extract_field(text, "Description"),
        "temperament": extract_field(text, "Temperament"),
        "life_span": extract_field(text, "Life span"),
        "weight_kg": weight_kg,
        "grooming_level": extract_int_field(text, "Grooming level"),
        "energy_level": extract_int_field(text, "Energy level"),
        "health_issues_score": extract_int_field(text, "Health issues score"),
        "child_friendly_score": extract_int_field(text, "Child friendly score"),
        "dog_friendly_score": extract_int_field(text, "Dog friendly score"),
        "stranger_friendly_score": extract_int_field(text, "Stranger friendly score"),
        "intelligence_score": extract_int_field(text, "Intelligence score"),
        "hairless": extract_field(text, "Hairless"),
        "shedding_level": extract_int_field(text, "Shedding level"),
        "social_needs_score": extract_int_field(text, "Social needs score"),
        "vocalisation_score": extract_int_field(text, "Vocalisation score"),
        "hypoallergenic": extract_int_field(text, "Hypoallergenic"),
        "wikipedia_url": metadata.get("wikipedia_url"),
        "reference_image_id": metadata.get("reference_image_id"),
        "image_url": metadata.get("image_url"),
        "raw_text": text,
    }


profiles = [build_structured_profile(chunk) for chunk in chunks]

print("Structured profiles:", len(profiles))
show_rows(profiles[:3], limit=3)

Structured profiles: 67
ROW 1
{
  "breed_id": "abys",
  "breed_name": "Abyssinian",
  "origin": "Egypt",
  "description": "The Abyssinian is easy to care for, and a joy to have in your home. They’re affectionate cats and love both people and other animals.",
  "temperament": "Active, Energetic, Independent, Intelligent, Gentle",
  "life_span": "14 - 15 years",
  "weight_kg": [
    3,
    5
  ],
  "grooming_level": 1,
  "energy_level": 5,
  "health_issues_score": 2,
  "child_friendly_score": 3,
  "dog_friendly_score": 4,
  "stranger_friendly_score": 5,
  "intelligence_score": 5,
  "hairless": "no",
  "shedding_level": 2,
  "social_needs_score": 5,
  "vocalisation_score": 1,
  "hypoallergenic": 0,
  "wikipedia_url": "https://en.wikipedia.org/wiki/Abyssinian_(cat)",
  "reference_image_id": "0XYvRd7oD",
  "image_url": null,
  "raw_text": "Breed name: Abyssinian\nDescription: The Abyssinian is easy to care for, and a joy to have in your home. They’re affectionate cats and love both people a

In [26]:
for breed in ["British Shorthair", "Maine Coon", "Siamese", "Persian", "Sphynx"]:
    print("=" * 100)
    profile = next((p for p in profiles if p["breed_name"] == breed), None)

    if profile is None:
        print("Not found:", breed)
        continue

    show_json({
        "breed_name": profile["breed_name"],
        "origin": profile["origin"],
        "temperament": profile["temperament"],
        "weight_kg": profile["weight_kg"],
        "grooming_level": profile["grooming_level"],
        "energy_level": profile["energy_level"],
        "shedding_level": profile["shedding_level"],
        "social_needs_score": profile["social_needs_score"],
        "vocalisation_score": profile["vocalisation_score"],
        "hairless": profile["hairless"],
        "description": preview_text(profile["description"], 500),
    })

{
  "breed_name": "British Shorthair",
  "origin": "United Kingdom",
  "temperament": "Affectionate, Easy Going, Gentle, Loyal, Patient, calm",
  "weight_kg": [
    5,
    9
  ],
  "grooming_level": 2,
  "energy_level": 2,
  "shedding_level": 4,
  "social_needs_score": 3,
  "vocalisation_score": 1,
  "hairless": "no",
  "description": "The British Shorthair is a very pleasant cat to have as a companion, ans is easy going and placid. The British is a fiercely loyal, loving cat and will attach herself to every one of her family members. While loving to play, she doesn't need hourly attention. If she is in the mood to play, she will find someone and bring a toy to that person. The British also plays well by herself, and thus is a good companion for single people."
}
{
  "breed_name": "Maine Coon",
  "origin": "United States",
  "temperament": "Adaptable, Intelligent, Loving, Gentle, Independent",
  "weight_kg": [
    5,
    8
  ],
  "grooming_level": 3,
  "energy_level": 3,
  "shedding_le

In [27]:
def structured_score(question: str, profile: Dict[str, Any]) -> int:
    q = question.lower().replace("ё", "е")
    score = 0

    text_blob = " ".join([
        str(profile.get("breed_name") or ""),
        str(profile.get("description") or ""),
        str(profile.get("temperament") or ""),
        str(profile.get("origin") or ""),
    ]).lower()

    # Базовый keyword overlap
    score += keyword_score(question, text_blob)

    # Размер / вес
    weight = profile.get("weight_kg")
    if weight:
        min_w, max_w = weight
        if any(word in q for word in ["large", "big", "гигант", "больш", "крупн"]):
            if max_w >= 8:
                score += 5
            elif max_w >= 6:
                score += 2

        if any(word in q for word in ["small", "tiny", "маленьк", "небольш"]):
            if max_w <= 5:
                score += 4

    # Характер
    temperament = str(profile.get("temperament") or "").lower()

    if any(word in q for word in ["gentle", "calm", "quiet", "спокойн", "тих", "ласков"]):
        for trait in ["gentle", "calm", "quiet", "easy going", "patient", "affectionate"]:
            if trait in temperament:
                score += 3

    if any(word in q for word in ["active", "energetic", "playful", "активн", "энергич", "игрив"]):
        if (profile.get("energy_level") or 0) >= 4:
            score += 4
        for trait in ["active", "energetic", "playful", "lively"]:
            if trait in temperament:
                score += 2

    if any(word in q for word in ["social", "friendly", "общительн", "дружелюб", "любит людей"]):
        if (profile.get("social_needs_score") or 0) >= 4:
            score += 3
        for trait in ["social", "friendly", "sociable", "loving"]:
            if trait in temperament:
                score += 2

    # Разговорчивость
    if any(word in q for word in ["vocal", "talkative", "разговорч", "болтлив", "мяука"]):
        if (profile.get("vocalisation_score") or 0) >= 4:
            score += 6

    # Интеллект
    if any(word in q for word in ["intelligent", "clever", "умн", "сообразительн"]):
        if (profile.get("intelligence_score") or 0) >= 4:
            score += 4

    # Уход / груминг
    if any(word in q for word in ["grooming", "care", "уход", "вычес", "шерсть"]):
        grooming = profile.get("grooming_level") or 0
        if grooming >= 4:
            score += 5
        elif grooming >= 2:
            score += 2

    # Линька
    if any(word in q for word in ["shedding", "линяет", "линька"]):
        if (profile.get("shedding_level") or 0) >= 4:
            score += 4

    # Голые / сфинксы
    if any(word in q for word in ["hairless", "без шерсти", "лыс", "гол"]):
        if str(profile.get("hairless")).lower() in {"yes", "1", "true"}:
            score += 8

    # Гипоаллергенность
    if any(word in q for word in ["hypoallergenic", "аллерг"]):
        if profile.get("hypoallergenic") == 1:
            score += 5

    return score


def structured_retrieve(question: str, top_k: int = 5) -> List[Dict[str, Any]]:
    scored = []

    for profile in profiles:
        score = structured_score(question, profile)
        scored.append({
            "score": score,
            "profile": profile,
        })

    scored = sorted(scored, key=lambda x: x["score"], reverse=True)
    return scored[:top_k]

In [29]:
structured_queries = [
    "large gentle cat",
    "большая спокойная кошка",
    "hairless cat care",
    "как ухаживать за сфинксом",
    "vocal intelligent cat",
    "разговорчивая умная кошка",
    "long hair grooming",
    "кошка с высоким уровнем ухода за шерстью",
]

for q in structured_queries:
    print("=" * 100)
    print("QUERY:", q)

    results = structured_retrieve(q, top_k=5)

    for i, item in enumerate(results, start=1):
        p = item["profile"]
        print("-" * 80)
        print("rank:", i)
        print("score:", item["score"])
        print("breed:", p["breed_name"])
        print("temperament:", p["temperament"])
        print("weight_kg:", p["weight_kg"])
        print("grooming:", p["grooming_level"])
        print("energy:", p["energy_level"])
        print("vocalisation:", p["vocalisation_score"])
        print("hairless:", p["hairless"])
        print("description:", preview_text(p["description"], 300))

QUERY: large gentle cat
--------------------------------------------------------------------------------
rank: 1
score: 21
breed: British Shorthair
temperament: Affectionate, Easy Going, Gentle, Loyal, Patient, calm
weight_kg: (5, 9)
grooming: 2
energy: 2
vocalisation: 1
hairless: no
description: The British Shorthair is a very pleasant cat to have as a companion, ans is easy going and placid. The British is a fiercely loyal, loving cat and will attach herself to every one of her family members. While loving to play, she doesn't need hourly attention. If she is in the mood to play, she will ...
--------------------------------------------------------------------------------
rank: 2
score: 15
breed: Ragamuffin
temperament: Affectionate, Friendly, Gentle, Calm
weight_kg: (4, 9)
grooming: 3
energy: 3
vocalisation: 1
hairless: no
description: The Ragamuffin is calm, even tempered and gets along well with all family members. Changes in routine generally do not upset her. She is an ideal com

## Hybrid retrieval v2: alias + structured fields

Теперь собираем более правильную retrieval-стратегию:

1. Сначала ищем явное упоминание породы через aliases.
2. Если порода найдена — возвращаем профиль этой породы.
3. Если порода не найдена — используем structured scoring по CatAPI fields.
4. Если все scores равны 0 — не возвращаем случайную породу, а честно говорим, что подходящий контекст не найден.

Это уже хороший baseline для backend до подключения embeddings.

In [30]:
chunk_by_breed = {
    chunk.get("breed_name"): chunk
    for chunk in chunks
}

profile_by_breed = {
    profile.get("breed_name"): profile
    for profile in profiles
}

print("Chunks by breed:", len(chunk_by_breed))
print("Profiles by breed:", len(profile_by_breed))

for breed in ["British Shorthair", "Maine Coon", "Sphynx", "Siamese", "Persian"]:
    print(breed, "→", breed in chunk_by_breed, breed in profile_by_breed)

Chunks by breed: 67
Profiles by breed: 67
British Shorthair → True True
Maine Coon → True True
Sphynx → True True
Siamese → True True
Persian → True True


In [31]:
def profile_to_chunk(profile: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    breed_name = profile.get("breed_name")
    return chunk_by_breed.get(breed_name)


def hybrid_retrieve_v2(question: str, top_k: int = 5) -> Dict[str, Any]:
    detected_breed = detect_breed_alias(question)

    # 1. Alias exact breed retrieval
    if detected_breed:
        chunk = chunk_by_breed.get(detected_breed)

        if chunk:
            return {
                "strategy": "alias_exact_breed",
                "detected_breed": detected_breed,
                "results": [
                    {
                        "rank": 1,
                        "score": None,
                        "chunk": chunk,
                    }
                ],
                "warning": None,
            }

        return {
            "strategy": "alias_detected_but_chunk_missing",
            "detected_breed": detected_breed,
            "results": [],
            "warning": f"Breed detected as {detected_breed}, but chunk was not found.",
        }

    # 2. Structured retrieval fallback
    structured_results = structured_retrieve(question, top_k=top_k)

    converted_results = []
    for item in structured_results:
        score = item["score"]

        # Не возвращаем мусор с нулевым score
        if score <= 0:
            continue

        profile = item["profile"]
        chunk = profile_to_chunk(profile)

        if chunk is None:
            continue

        converted_results.append({
            "rank": len(converted_results) + 1,
            "score": score,
            "chunk": chunk,
        })

    if not converted_results:
        return {
            "strategy": "structured_no_match",
            "detected_breed": None,
            "results": [],
            "warning": "No relevant CatAPI context found with current retrieval rules.",
        }

    return {
        "strategy": "structured_fields",
        "detected_breed": None,
        "results": converted_results,
        "warning": None,
    }

In [32]:
def print_retrieval_result_v2(question: str, result: Dict[str, Any]):
    print("=" * 100)
    print("QUESTION:", question)
    print("STRATEGY:", result.get("strategy"))
    print("DETECTED BREED:", result.get("detected_breed"))

    if result.get("warning"):
        print("WARNING:", result["warning"])

    if not result.get("results"):
        print("No retrieved context.")
        return

    for item in result["results"]:
        chunk = item["chunk"]
        metadata = chunk.get("metadata", {}) or {}

        print("-" * 80)
        print("rank:", item["rank"])
        print("score:", item["score"])
        print("breed:", chunk.get("breed_name"))
        print("breed_id:", chunk.get("breed_id"))
        print("origin:", metadata.get("origin"))
        print("wikipedia_url:", metadata.get("wikipedia_url"))
        print("reference_image_id:", metadata.get("reference_image_id"))
        print("image_url:", metadata.get("image_url"))
        print("text:", preview_text(chunk.get("text", ""), 450))

In [33]:
hybrid_v2_queries = [
    "Чем британские котики отличаются от обычных?",
    "Расскажи про мейн куна",
    "Как ухаживать за сфинксом?",
    "Сиамская кошка разговорчивая и умная?",
    "Persian cat grooming",
    "large gentle cat",
    "большая спокойная кошка",
    "hairless cat care",
    "vocal intelligent cat",
    "разговорчивая умная кошка",
    "long hair grooming",
    "кошка с высоким уровнем ухода за шерстью",
    "какая кошка умеет чинить ноутбук?",
]

for q in hybrid_v2_queries:
    result = hybrid_retrieve_v2(q, top_k=5)
    print_retrieval_result_v2(q, result)

QUESTION: Чем британские котики отличаются от обычных?
STRATEGY: alias_exact_breed
DETECTED BREED: British Shorthair
--------------------------------------------------------------------------------
rank: 1
score: None
breed: British Shorthair
breed_id: bsho
origin: United Kingdom
wikipedia_url: https://en.wikipedia.org/wiki/British_Shorthair
reference_image_id: s4wQfYoEk
image_url: None
text: Breed name: British Shorthair Description: The British Shorthair is a very pleasant cat to have as a companion, ans is easy going and placid. The British is a fiercely loyal, loving cat and will attach herself to every one of her family members. While loving to play, she doesn't need hourly attention. If she is in the mood to play, she will find someone and bring a toy to that person. The British also plays well by herself, and thus is a good com...
QUESTION: Расскажи про мейн куна
STRATEGY: alias_exact_breed
DETECTED BREED: Maine Coon
--------------------------------------------------------------

## Вывод после Hybrid retrieval v2

Теперь retrieval работает намного лучше:

- запросы с явной породой обрабатываются через alias detection;
- описательные запросы обрабатываются через structured CatAPI fields;
- если совпадений нет, система не возвращает случайную породу;
- retrieved context явно виден до генерации ответа.

Это хороший baseline для приложения.

Embeddings и ChromaDB можно добавить позже, но уже сейчас понятно, что для русскоязычного сервиса полезна гибридная схема:

`breed alias detection → structured retrieval → vector retrieval`

In [34]:
def build_rag_prompt_v2(question: str, retrieved_result: Dict[str, Any]) -> str:
    context_blocks = []

    for item in retrieved_result.get("results", []):
        chunk = item["chunk"]
        metadata = chunk.get("metadata", {}) or {}

        block = f"""
[Context chunk]
Rank: {item.get("rank")}
Retrieval strategy: {retrieved_result.get("strategy")}
Retrieval score: {item.get("score")}
Breed: {chunk.get("breed_name")}
Breed ID: {chunk.get("breed_id")}
Origin: {metadata.get("origin")}
Wikipedia URL: {metadata.get("wikipedia_url")}
Reference image ID: {metadata.get("reference_image_id")}
Image URL: {metadata.get("image_url")}

Text:
{chunk.get("text")}
""".strip()

        context_blocks.append(block)

    if context_blocks:
        context_text = "\n\n---\n\n".join(context_blocks)
    else:
        context_text = "No relevant CatAPI context was retrieved."

    prompt = f"""
SYSTEM INSTRUCTION:
{SYSTEM_INSTRUCTION}

RETRIEVED CONTEXT:
{context_text}

USER QUESTION:
{question}

ANSWER:
""".strip()

    return prompt

In [35]:
prompt_questions = [
    "Чем британские котики отличаются от обычных?",
    "Какая кошка большая, спокойная и ласковая?",
    "Какая кошка разговорчивая и умная?",
    "Как ухаживать за сфинксом?",
    "Какая кошка умеет чинить ноутбук?",
]

for q in prompt_questions:
    print("=" * 120)
    print("QUESTION:", q)

    retrieved = hybrid_retrieve_v2(q, top_k=3)
    print("STRATEGY:", retrieved["strategy"])
    print("DETECTED BREED:", retrieved["detected_breed"])
    print("WARNING:", retrieved["warning"])

    print("\nPROMPT:")
    print(build_rag_prompt_v2(q, retrieved))
    print()

QUESTION: Чем британские котики отличаются от обычных?
STRATEGY: alias_exact_breed
DETECTED BREED: British Shorthair

PROMPT:
SYSTEM INSTRUCTION:
Ты — дружелюбный ассистент-консультант по породам кошек.

Отвечай на русском языке.
Используй только факты из retrieved context.
Если информации мало, честно скажи, что данных недостаточно.
Не давай категоричных медицинских советов.
Если вопрос касается здоровья, мягко посоветуй обратиться к ветеринару.

RETRIEVED CONTEXT:
[Context chunk]
Rank: 1
Retrieval strategy: alias_exact_breed
Retrieval score: None
Breed: British Shorthair
Breed ID: bsho
Origin: United Kingdom
Wikipedia URL: https://en.wikipedia.org/wiki/British_Shorthair
Reference image ID: s4wQfYoEk
Image URL: None

Text:
Breed name: British Shorthair
Description: The British Shorthair is a very pleasant cat to have as a companion, ans is easy going and placid. The British is a fiercely loyal, loving cat and will attach herself to every one of her family members. While loving to play

In [36]:
# Уберём слишком широкий alias.
# "hairless" — это не конкретная порода, а признак.
# Поэтому такие запросы лучше обрабатывать через structured retrieval.

BREED_ALIASES.pop("hairless", None)

for q in [
    "hairless cat care",
    "Sphynx care",
    "как ухаживать за сфинксом",
]:
    print(q, "→", detect_breed_alias(q))

hairless cat care → None
Sphynx care → Sphynx
как ухаживать за сфинксом → Sphynx


In [37]:
for q in [
    "hairless cat care",
    "hairless friendly cat",
    "как ухаживать за сфинксом",
]:
    result = hybrid_retrieve_v2(q, top_k=5)
    print_retrieval_result_v2(q, result)

QUESTION: hairless cat care
STRATEGY: structured_fields
DETECTED BREED: None
--------------------------------------------------------------------------------
rank: 1
score: 10
breed: Donskoy
breed_id: dons
origin: Russia
wikipedia_url: https://en.wikipedia.org/wiki/Donskoy_(cat)
reference_image_id: 3KG57GfMW
image_url: None
text: Breed name: Donskoy Description: Donskoy are affectionate, intelligent, and easy-going. They demand lots of attention and interaction. The Donskoy also gets along well with other pets. It is now thought the same gene that causes degrees of hairlessness in the Donskoy also causes alterations in cat personality, making them calmer the less hair they have. Temperament: Playful, affectionate, loyal, social Origin: Russia Life span: 12 - 15 years Weig...
--------------------------------------------------------------------------------
rank: 2
score: 10
breed: Sphynx
breed_id: sphy
origin: Canada
wikipedia_url: https://en.wikipedia.org/wiki/Sphynx_(cat)
reference_ima

## Baseline RAG conclusions

На этом этапе у нас есть рабочий baseline retrieval без embeddings и ChromaDB.

Что работает хорошо:

- если пользователь явно называет породу, срабатывает `alias_exact_breed`;
- если пользователь описывает желаемые признаки, срабатывает `structured_fields`;
- если вопрос не относится к породам кошек, система не возвращает случайный chunk;
- retrieved context явно виден до генерации ответа;
- final RAG prompt можно проверить глазами.

Это уже полезная RAG-архитектура:

`question → breed alias / structured retrieval → retrieved context → prompt → answer`

Ограничения:

- structured rules написаны руками;
- semantic similarity пока не используется;
- русские описательные запросы требуют ручной поддержки словарей;
- CatAPI даёт ограниченное описание пород;
- image_url пока пустой, но есть reference_image_id.

Следующий слой: попробовать embeddings in memory без ChromaDB.

## Optional: semantic retrieval with embeddings

Теперь попробуем semantic retrieval без ChromaDB.

Мы не строим persistent index и не используем vector database.

Идея:

1. Загружаем embedding model.
2. Считаем embeddings для 67 chunks.
3. Считаем embedding для query.
4. Ищем chunks с максимальной cosine similarity.

Если `sentence-transformers` не работает в окружении, эту часть можно пропустить.

In [38]:
EMBEDDINGS_AVAILABLE = False
embedding_model = None

ALLOW_MODEL_DOWNLOAD = True  # если модель ещё не скачана, разрешаем загрузку

try:
    from sentence_transformers import SentenceTransformer
    import numpy as np

    MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

    if ALLOW_MODEL_DOWNLOAD:
        embedding_model = SentenceTransformer(MODEL_NAME)
    else:
        embedding_model = SentenceTransformer(MODEL_NAME, local_files_only=True)

    EMBEDDINGS_AVAILABLE = True
    print("Embedding model loaded:", MODEL_NAME)

except Exception as e:
    print("Embeddings are not available in this environment.")
    print("Reason:", repr(e))
    print()
    print("You can skip embedding cells and continue with hybrid_retrieve_v2.")

Embeddings are not available in this environment.
Reason: ModuleNotFoundError("Could not import module 'AutoModelForSequenceClassification'. Are this object's requirements defined correctly?")

You can skip embedding cells and continue with hybrid_retrieve_v2.


In [39]:
if EMBEDDINGS_AVAILABLE:
    chunk_texts = [chunk["text"] for chunk in chunks]
    chunk_ids = [chunk["chunk_id"] for chunk in chunks]

    chunk_embeddings = embedding_model.encode(
        chunk_texts,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    print("Chunks:", len(chunk_texts))
    print("Embeddings shape:", chunk_embeddings.shape)
else:
    print("Skipping embeddings: model is not available.")

Skipping embeddings: model is not available.


In [40]:
def semantic_retrieve(question: str, top_k: int = 5) -> Dict[str, Any]:
    if not EMBEDDINGS_AVAILABLE:
        return {
            "strategy": "semantic_unavailable",
            "detected_breed": None,
            "results": [],
            "warning": "Embedding model is not available.",
        }

    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True,
    )[0]

    # cosine similarity because embeddings are normalized
    scores = chunk_embeddings @ query_embedding

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "score": float(scores[idx]),
            "chunk": chunks[int(idx)],
        })

    return {
        "strategy": "semantic_in_memory",
        "detected_breed": None,
        "results": results,
        "warning": None,
    }

In [41]:
semantic_queries = [
    "British Shorthair round face dense coat",
    "large gentle cat",
    "hairless cat care",
    "vocal intelligent cat",
    "long hair grooming",
    "британская кошка круглая морда плотная шерсть",
    "мейн кун большой спокойный кот",
    "как ухаживать за сфинксом",
    "сиамская кошка разговорчивая",
]

for q in semantic_queries:
    result = semantic_retrieve(q, top_k=5)
    print_retrieval_result_v2(q, result)

QUESTION: British Shorthair round face dense coat
STRATEGY: semantic_unavailable
DETECTED BREED: None
No retrieved context.
QUESTION: large gentle cat
STRATEGY: semantic_unavailable
DETECTED BREED: None
No retrieved context.
QUESTION: hairless cat care
STRATEGY: semantic_unavailable
DETECTED BREED: None
No retrieved context.
QUESTION: vocal intelligent cat
STRATEGY: semantic_unavailable
DETECTED BREED: None
No retrieved context.
QUESTION: long hair grooming
STRATEGY: semantic_unavailable
DETECTED BREED: None
No retrieved context.
QUESTION: британская кошка круглая морда плотная шерсть
STRATEGY: semantic_unavailable
DETECTED BREED: None
No retrieved context.
QUESTION: мейн кун большой спокойный кот
STRATEGY: semantic_unavailable
DETECTED BREED: None
No retrieved context.
QUESTION: как ухаживать за сфинксом
STRATEGY: semantic_unavailable
DETECTED BREED: None
No retrieved context.
QUESTION: сиамская кошка разговорчивая
STRATEGY: semantic_unavailable
DETECTED BREED: None
No retrieved conte

In [42]:
compare_queries = [
    "Чем британские котики отличаются от обычных?",
    "Какая кошка большая, спокойная и ласковая?",
    "Какая кошка разговорчивая и умная?",
    "Как ухаживать за сфинксом?",
    "hairless cat care",
    "long hair grooming",
]

for q in compare_queries:
    print("=" * 120)
    print("QUESTION:", q)

    print("\nHYBRID V2")
    hybrid_result = hybrid_retrieve_v2(q, top_k=3)
    for item in hybrid_result.get("results", []):
        print(
            item["rank"],
            item["score"],
            item["chunk"].get("breed_name"),
        )
    if hybrid_result.get("warning"):
        print("WARNING:", hybrid_result["warning"])

    print("\nSEMANTIC")
    semantic_result = semantic_retrieve(q, top_k=3)
    for item in semantic_result.get("results", []):
        print(
            item["rank"],
            round(item["score"], 4) if item["score"] is not None else None,
            item["chunk"].get("breed_name"),
        )
    if semantic_result.get("warning"):
        print("WARNING:", semantic_result["warning"])

    print()

QUESTION: Чем британские котики отличаются от обычных?

HYBRID V2
1 None British Shorthair

SEMANTIC

QUESTION: Какая кошка большая, спокойная и ласковая?

HYBRID V2
1 20 British Shorthair
2 14 Ragamuffin
3 14 Ragdoll

SEMANTIC

QUESTION: Какая кошка разговорчивая и умная?

HYBRID V2
1 10 Balinese
2 10 Bengal
3 10 Bombay

SEMANTIC

QUESTION: Как ухаживать за сфинксом?

HYBRID V2
1 None Sphynx

SEMANTIC

QUESTION: hairless cat care

HYBRID V2
1 10 Donskoy
2 10 Sphynx
3 9 Bambino

SEMANTIC

QUESTION: long hair grooming

HYBRID V2
1 5 British Longhair
2 5 Chantilly-Tiffany
3 5 Himalayan

SEMANTIC

